# 🏆 TEKNOFEST 2025 - Hepsiburada Address Resolution Hackathon

Bu notebook, Kaggle aşaması için **Hashing/TF-IDF + LinearSVC** tabanlı bir baseline model sunar.

Adımlar:
1. Dosyaları yükle (`train.csv`, `test.csv`, `sample_submission.csv`).
2. Normalizasyon & kısaltma genişletme.
3. Model eğitimi (Hashing veya TF-IDF).
4. Kaggle’a yüklenebilir `submission.csv` üretimi.


In [3]:
# 📂 Gerekli kütüphaneler
import pandas as pd
import numpy as np
import re, unicodedata
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.feature_extraction.text import HashingVectorizer, TfidfVectorizer
from datetime import datetime

print("Hazır!")

Hazır!


In [2]:
# 📥 Dosyaları yükle (Colab menüsünden upload ile)
from google.colab import files
uploaded = files.upload()  # train.csv, test.csv, sample_submission.csv yükleyin


Saving sample_submission.csv to sample_submission.csv
Saving test.csv to test.csv
Saving train.csv to train.csv


In [4]:
# 🔤 Normalizasyon & Kısaltma
PUNCT_TABLE = str.maketrans({c: " " for c in r"""!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~"""})
ABBR_MAP = {
    "cad": "caddesi", "cad.": "caddesi", "cd": "caddesi", "cd.": "caddesi",
    "sok": "sokak", "sok.": "sokak", "sk": "sokak", "sk.": "sokak",
    "mh": "mahallesi", "mh.": "mahallesi", "mah": "mahallesi", "mah.": "mahallesi",
    "bulv": "bulvarı", "bulv.": "bulvarı", "blv": "bulvarı", "blv.": "bulvarı",
    "apt": "apartmanı", "apt.": "apartmanı",
    "no": "no", "no.": "no",
    "dr": "daire", "dr.": "daire", "d.": "daire",
    "k": "kat", "kat.": "kat",
    "i̇stanbul": "istanbul",
}

def normalize_tr(text: str) -> str:
    if not isinstance(text, str):
        return ""
    t = text.lower()
    t = unicodedata.normalize("NFKC", t)
    t = t.translate(PUNCT_TABLE)
    tokens = []
    for tok in t.split():
        tok_clean = tok.strip(".")
        tokens.append(ABBR_MAP.get(tok_clean, tok_clean))
    return " ".join(tokens)


In [5]:
# 📊 Verileri oku ve normalize et
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_sub = pd.read_csv("sample_submission.csv")

train_texts = train["address"].astype(str).apply(normalize_tr)
test_texts = test["address"].astype(str).apply(normalize_tr)
y = train["label"].values

print("Train size:", train.shape, " Test size:", test.shape)

Train size: (848237, 2)  Test size: (217241, 2)


In [ ]:
# ⚡ Hashing + LinearSVC (Hızlı Baseline)
vec = HashingVectorizer(analyzer="char", ngram_range=(3,5), n_features=2**20, alternate_sign=False, norm="l2")
X_train = vec.transform(train_texts)
X_test = vec.transform(test_texts)

X_tr, X_val, y_tr, y_val = train_test_split(X_train, y, test_size=0.15, stratify=y, random_state=42)
clf = LinearSVC(C=1.0)
clf.fit(X_tr, y_tr)
val_preds = clf.predict(X_val)
print("Validation Macro-F1:", f1_score(y_val, val_preds, average="macro"))

In [ ]:
# 🏁 Full train ile eğit ve test için tahmin yap
clf_full = LinearSVC(C=1.0)
clf_full.fit(X_train, y)
test_preds = clf_full.predict(X_test)

submission = pd.DataFrame({"id": test["id"], "label": test_preds.astype(int)})
out_name = f"submission_hash_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
submission.to_csv(out_name, index=False)
print("Saved:", out_name)

In [ ]:
# 🐢 Opsiyonel: TF-IDF (daha güçlü, daha ağır)
vec2 = TfidfVectorizer(analyzer="char", ngram_range=(3,5), min_df=3, max_features=1_000_000)
X_train2 = vec2.fit_transform(train_texts)
X_test2 = vec2.transform(test_texts)

clf2 = LinearSVC(C=1.0)
clf2.fit(X_train2, y)
test_preds2 = clf2.predict(X_test2)

submission2 = pd.DataFrame({"id": test["id"], "label": test_preds2.astype(int)})
out_name2 = f"submission_tfidf_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
submission2.to_csv(out_name2, index=False)
print("Saved:", out_name2)